# Explorer 07 - Artist clustering

**Purpose.** Build artist feature vectors, add node2vec graph embeddings, reduce with UMAP, and cluster with HDBSCAN.

**Feeds main report section:** 5.6 Artist clustering

**Modules used:** `vinyl.clustering`, `vinyl.networks`

**Tables read:** `release_artists`, `release_styles`, `release_formats`, `release_labels`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

**Status.** `clustering.node2vec_embeddings` and `clustering.combine_features` are stubs that raise `NotImplementedError`. Sections 3 and 4 will not run until they are written.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-07-clustering")
from vinyl import clustering, networks

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_styles = ingest.read_table(spark, snapshot, "release_styles")
release_formats = ingest.read_table(spark, snapshot, "release_formats")
release_labels = ingest.read_table(spark, snapshot, "release_labels")

## 1. Artist profile features (Spark)

In [ ]:
real_artists = release_artists.where(~F.col("artist_id").isin(list(networks.PLACEHOLDER_ARTIST_IDS)))
profile = clustering.artist_profile(real_artists, release_formats, release_labels, min_releases=5).cache()
print(profile.count(), "artists kept")
profile.describe().show()

## 2. Style TF-IDF (Spark)

In [ ]:
kept = real_artists.join(profile.select("artist_id"), "artist_id", "left_semi")
tfidf = clustering.style_tfidf(kept, release_styles, min_artists=20)
tfidf.orderBy(F.desc("tfidf")).show(20, truncate=False)

## 3. node2vec embeddings (stub)

In [ ]:
collab = networks.collaboration_edges(kept, min_weight=2).toPandas()
embeddings = clustering.node2vec_embeddings(collab, dimensions=64)

## 4. Combine, reduce, cluster

In [ ]:
features, artist_ids = clustering.combine_features(profile.toPandas(), tfidf.toPandas(), embeddings)
embedding_2d = clustering.reduce_umap(features, n_components=2)
embedding_10d = clustering.reduce_umap(features, n_components=10)
cluster_labels = clustering.cluster_hdbscan(embedding_10d, min_cluster_size=50)

In [ ]:
# TODO: scatter plot of embedding_2d coloured by cluster.
# TODO: describe each cluster: top styles, format mix, typical career span, example artists.
# TODO: share of artists labelled noise (-1); tune min_cluster_size.
# TODO: compare clusters with the Leiden communities from explorer 06.

## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] Minimum releases per artist and the number of artists clustered.
- [ ] node2vec library and parameters.
- [ ] Feature scaling and block weights.
- [ ] UMAP and HDBSCAN parameters, number of clusters, noise share.